In [ ]:
# ==============================
# 1️⃣ Setup
# ==============================
!pip install -U ultralytics


In [ ]:
# =============================
# 1️⃣ Setup & Check Dataset
# =============================
!pip install -U ultralytics

import os, shutil
from PIL import Image
from ultralytics import YOLO

# Check dataset structure
!ls /kaggle/input/safetyeye/data

# -------------------------------
# 2️⃣ Paths
# -------------------------------
src_train_img = "/kaggle/input/safetyeye/data/train/images"
src_train_lbl = "/kaggle/input/safetyeye/data/train/labels"

src_val_img   = "/kaggle/input/safetyeye/data/valid/images"
src_val_lbl   = "/kaggle/input/safetyeye/data/valid/labels"

src_test_img  = "/kaggle/input/safetyeye/data/test/images"
src_test_lbl  = "/kaggle/input/safetyeye/data/test/labels"

# Writable copies
dst_base = "/kaggle/working/safetyeye"
dst_train_img, dst_train_lbl = os.path.join(dst_base,"train/images"), os.path.join(dst_base,"train/labels")
dst_val_img,   dst_val_lbl   = os.path.join(dst_base,"valid/images"), os.path.join(dst_base,"valid/labels")
dst_test_img,  dst_test_lbl  = os.path.join(dst_base,"test/images"),  os.path.join(dst_base,"test/labels")

for p in [dst_train_img,dst_train_lbl,dst_val_img,dst_val_lbl,dst_test_img,dst_test_lbl]:
    os.makedirs(p, exist_ok=True)

# -------------------------------
# 3️⃣ Copy Dataset
# -------------------------------
def copy_folder(src, dst):
    for f in os.listdir(src):
        shutil.copy(os.path.join(src,f), dst)

copy_folder(src_train_img, dst_train_img)
copy_folder(src_train_lbl, dst_train_lbl)
copy_folder(src_val_img, dst_val_img)
copy_folder(src_val_lbl, dst_val_lbl)
copy_folder(src_test_img, dst_test_img)
copy_folder(src_test_lbl, dst_test_lbl)

print("✅ Dataset copied to working directory")

# -------------------------------
# 4️⃣ Clean Dataset
# -------------------------------
mapping = {6: 3, 9: 4}  # remap classes if needed
corrupt_count, fixed_count = 0, 0

for lbl_file in os.listdir(dst_train_lbl):
    lbl_path = os.path.join(dst_train_lbl, lbl_file)
    img_file = lbl_file.replace(".txt", ".jpg")  # adjust extension if .png
    img_path = os.path.join(dst_train_img, img_file)

    # Remove if image missing
    if not os.path.exists(img_path):
        os.remove(lbl_path)
        corrupt_count += 1
        continue

    # Remove if corrupted
    try:
        img = Image.open(img_path)
        img.verify()
    except:
        os.remove(img_path)
        os.remove(lbl_path)
        corrupt_count += 1
        continue

    # Fix class IDs
    with open(lbl_path, "r") as f:
        lines = f.readlines()
    new_lines, changed = [], False
    for line in lines:
        parts = line.strip().split()
        if not parts: continue
        cid = int(parts[0])
        if cid in mapping:
            parts[0] = str(mapping[cid])
            changed = True
        new_lines.append(" ".join(parts))
    if changed:
        with open(lbl_path, "w") as f:
            f.write("\n".join(new_lines))
        fixed_count += 1

print(f"✅ Cleaning done: {fixed_count} labels fixed, {corrupt_count} corrupted removed")

# -------------------------------
# 5️⃣ Create data.yaml
# -------------------------------
data_yaml = "/kaggle/working/data.yaml"
with open(data_yaml, "w") as f:
    f.write(f"""
train: {dst_train_img}
val: {dst_val_img}
test: {dst_test_img}
nc: 5
names: [person, helmet, vest, machinery, cone]
""")
print("✅ data.yaml created")

# -------------------------------
# 6️⃣ Train YOLOv8 (Improved)
# -------------------------------
model = YOLO("yolov8s.pt")  # use small model for higher accuracy

model.train(
    data=data_yaml,
    epochs=100,        # train longer
    imgsz=640,
    batch=16,
    optimizer="AdamW", # better optimizer
    lr0=0.001,         # lower LR improves stability
    device=0,          # GPU
    augment=True,      # stronger augmentation
    name="safetyeye_v8s_improved"
)

# -------------------------------
# 7️⃣ Evaluate Model
# -------------------------------
metrics = model.val()
print("📊 Final Evaluation Metrics:", metrics)


In [ ]:
!ls /kaggle/input

In [ ]:
!pip install -U ultralytics


In [ ]:
from ultralytics import YOLO


In [3]:
# =============================
# 0️⃣ Install YOLO package
# =============================
!pip install -U ultralytics

# =============================
# 1️⃣ Imports
# =============================
from ultralytics import YOLO
import os

# =============================
# 2️⃣ Paths
# =============================
model_path = "/kaggle/input/best-pt/best.pt"   # Corrected path

test_images_path = "/kaggle/working/test_images/"   # Put your sample images here
output_path = "/kaggle/working/violation_outputs/"

os.makedirs(test_images_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

# =============================
# 3️⃣ Load YOLO model
# =============================
model = YOLO(model_path)

# =============================
# 4️⃣ Violation check function
# =============================
def check_violations(detections):
    violations = []
    for det in detections:
        for cls, conf, box in zip(det.boxes.cls, det.boxes.conf, det.boxes.xyxy):
            cls_name = model.names[int(cls)]
            if cls_name == "person":
                violations.append("Person detected")
            elif cls_name == "helmet":
                violations.append("Helmet detected")
            elif cls_name == "vest":
                violations.append("Vest detected")
    return violations

# =============================
# 5️⃣ Run detection on test images
# =============================
image_files = os.listdir(test_images_path)
if not image_files:
    print("⚠️ No images found in test_images folder. Upload some sample images first.")

for img_name in image_files:
    img_path = os.path.join(test_images_path, img_name)
    results = model(img_path)

    # Check for violations
    alerts = check_violations(results)

    if alerts:
        print(f"⚠️ {img_name}: {alerts}")
    else:
        print(f"✅ {img_name}: No violations")

    # Save output image with detection boxes
    output_img_path = os.path.join(output_path, img_name)
    results[0].plot(save_path=output_img_path)

print(f"\nAll outputs saved in: {output_path}")


  Using cached ultralytics-8.3.203-py3-none-any.whl.metadata (37 kB)
  Using cached ultralytics_thop-2.0.17-py3-none-any.whl.metadata (14 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
Using cached ultralytics-8.3.203-py3-none-any.whl (1.1 MB)
Using cached nvidia_c

In [5]:
import os
print(os.listdir("/kaggle/input/"))


['safety1-jpeg', 'best-pt', 'test-videos']


In [6]:
import os, shutil

os.makedirs("/kaggle/working/test_images/", exist_ok=True)

# Move all uploaded images to the test_images folder
for f in os.listdir("/kaggle/working/"):
    if f.endswith((".jpg", ".png", ".jpeg")):
        shutil.move(f"/kaggle/working/{f}", "/kaggle/working/test_images/")


In [7]:
import os
print(os.listdir("/kaggle/input/"))

['safety1-jpeg', 'best-pt', 'test-videos']


In [8]:
# =============================
# 0️⃣ Install YOLO package
# =============================
!pip install -U ultralytics

# =============================
# 1️⃣ Imports
# =============================
from ultralytics import YOLO
import os
import shutil

# =============================
# 2️⃣ Paths
# =============================
model_path = "/kaggle/input/best-pt/best.pt"        # Correct model path
input_images_folder = "/kaggle/input/safety1-jpeg" # Folder with uploaded test images
test_images_path = "/kaggle/working/test_images/"  # Notebook working folder
output_path = "/kaggle/working/violation_outputs/"

os.makedirs(test_images_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

# Copy images from dataset input to working folder
for f in os.listdir(input_images_folder):
    if f.endswith((".jpg", ".jpeg", ".png")):
        shutil.copy(os.path.join(input_images_folder, f), test_images_path)

# =============================
# 3️⃣ Load YOLO model
# =============================
model = YOLO(model_path)

# =============================
# 4️⃣ Violation check function
# =============================
def check_violations(detections):
    violations = []
    for det in detections:
        for cls, conf, box in zip(det.boxes.cls, det.boxes.conf, det.boxes.xyxy):
            cls_name = model.names[int(cls)]
            if cls_name == "person":
                violations.append("Person detected")
            elif cls_name == "helmet":
                violations.append("Helmet detected")
            elif cls_name == "vest":
                violations.append("Vest detected")
    return violations

# =============================
# 5️⃣ Run detection on test images
# =============================
image_files = os.listdir(test_images_path)
if not image_files:
    print("⚠️ No images found in test_images folder. Upload some sample images first.")

for img_name in image_files:
    img_path = os.path.join(test_images_path, img_name)
    results = model(img_path)

    # Check for violations
    alerts = check_violations(results)

    if alerts:
        print(f"⚠️ {img_name}: {alerts}")
    else:
        print(f"✅ {img_name}: No violations")



image 1/1 /kaggle/working/test_images/safety1.jpeg: 768x768 25 persons, 8 no_helmets, 14 vests, 2 cones, 779.1ms
Speed: 19.6ms preprocess, 779.1ms inference, 32.4ms postprocess per image at shape (1, 3, 768, 768)
⚠️ safety1.jpeg: ['Person detected', 'Person detected', 'Person detected', 'Person detected', 'Person detected', 'Person detected', 'Vest detected', 'Person detected', 'Person detected', 'Person detected', 'Person detected', 'Person detected', 'Person detected', 'Vest detected', 'Vest detected', 'Vest detected', 'Vest detected', 'Person detected', 'Vest detected', 'Vest detected', 'Person detected', 'Vest detected', 'Vest detected', 'Person detected', 'Person detected', 'Person detected', 'Person detected', 'Person detected', 'Person detected', 'Vest detected', 'Person detected', 'Person detected', 'Vest detected', 'Vest detected', 'Person detected', 'Person detected', 'Person detected', 'Vest detected', 'Vest detected']


In [9]:
import os
import shutil
from ultralytics import YOLO

# =============================
# Paths
# =============================
dataset_model_folder = "/kaggle/input/best-pt"   # Kaggle dataset folder
model_working_path = "/kaggle/working/best.pt"   # Copy here to load

# Copy model to working folder
shutil.copy(os.path.join(dataset_model_folder, "best.pt"), model_working_path)

# Load the YOLO model
model = YOLO(model_working_path)

# Test image
test_image = "/kaggle/input/safety1-jpeg/safety1.jpeg"  # you can also copy it to working if you want
results = model.predict(source=test_image, save=True)

print("✅ Detection completed! Check runs/detect/ folder for output images.")



image 1/1 /kaggle/input/safety1-jpeg/safety1.jpeg: 768x768 25 persons, 8 no_helmets, 14 vests, 2 cones, 629.0ms
Speed: 5.8ms preprocess, 629.0ms inference, 8.2ms postprocess per image at shape (1, 3, 768, 768)
Results saved to /kaggle/working/runs/detect/predict
✅ Detection completed! Check runs/detect/ folder for output images.


In [10]:
from ultralytics import YOLO
import os

model_path = "/kaggle/input/best-pt/best.pt"

# Load YOLO model
model = YOLO(model_path)

# Run detection on video (outputs automatically saved to runs/detect/predict)
video_input_path = "/kaggle/input/test-videos/safety_video.mp4"
results = model(video_input_path, save=True)

# Check output files
output_files = os.listdir("/kaggle/working/runs/detect/predict")
print("Output files:", output_files)



WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/490) /kaggle/input/test-videos/safety_video.mp4: 448x768 6 persons, 3 no_helmets, 1 vest, 369.3ms
video 1/1 (frame 2/490) /kaggle/input/test-videos/safety_video.mp4: 448x768 6 persons, 3 no_helmets, 1 vest, 358.9ms
video 1/1 (frame 3/490) /kaggle/input/test-videos/safety_video.mp4: 448x768 6 persons, 3 no_helmets, 2 vests, 363.7ms
video 1/1 (frame 4/490) /kaggle/input/test-videos/safety_video.mp4: 448x768 7 persons, 3 no_helmets, 1 v

In [1]:
import os
os.listdir("/kaggle/input")


['safety1-jpeg', 'best-pt', 'test-videos']